# Материалы статьи по широтным зонам и прогнозу foF2

Ноутбук собирает таблицы, графики Plotly и черновые материалы статьи по сравнению точности 24-часового прогноза foF2 между широтными зонами.

Основные этапы:

1. Загружаются готовые таблицы с ошибками моделей по станциям и зонам.
2. Строятся графики в едином оформлении для статьи.
3. Формируется документ с текстом, таблицами и рисунками.

Этот ноутбук не обучает модели заново: он использует уже рассчитанные результаты.

In [ ]:
# Подготовка библиотек, путей к данным и общих настроек ноутбука.
from pathlib import Path
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from docx import Document
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.shared import Cm, Pt

ROOT = Path.cwd()
REPORT_DIR = ROOT / "reports" / "latitude_zone_analysis"
OUT_DIR = ROOT / "reports" / "article_gost_materials"
OUT_DIR.mkdir(parents=True, exist_ok=True)

ZONE_ORDER = ["Low", "S_mid", "N_mid", "N_high"]
ZONE_LABELS = {
    "Low": "?????? ??????",
    "S_mid": "????? ???????",
    "N_mid": "???????? ???????",
    "N_high": "???????? ???????",
}
MODEL_ORDER = ["LinearRegression", "ElasticNet", "RandomForest", "XGBoost", "CatBoost"]
NONLINEAR_MODELS = ["RandomForest", "XGBoost", "CatBoost"]
COLORS = {
    "LinearRegression": "#4C78A8",
    "ElasticNet": "#F58518",
    "RandomForest": "#54A24B",
    "XGBoost": "#E45756",
    "CatBoost": "#72B7B2",
    "MAE": "#4C78A8",
    "RMSE": "#E45756",
    "R?": "#54A24B",
    "??????????": "#F58518",
}


In [ ]:
# Загрузка готовых таблиц с метриками по станциям, моделям и широтным зонам.
station_model = pd.read_csv(REPORT_DIR / "station_latitude_zone_model_errors.csv")
zone_model = pd.read_csv(REPORT_DIR / "zone_model_summary.csv")
zone_summary_nl = pd.read_csv(REPORT_DIR / "zone_summary_nonlinear_models.csv", index_col=0)
metadata = pd.read_csv(ROOT / "reports" / "fof2_fast_ml_24h_station_metadata.csv")

zone_summary_nl.loc[ZONE_ORDER]


In [ ]:
# Функция задает единый стиль графиков для статьи: шрифт, сетку, легенду и размеры.
def gost_layout(fig, title, y_title=None, x_title="???????? ????", legend_y=-0.24):
    fig.update_layout(
        title={"text": title, "x": 0.5, "xanchor": "center"},
        font={"family": "Times New Roman", "size": 14, "color": "black"},
        plot_bgcolor="white",
        paper_bgcolor="white",
        width=1100,
        height=650,
        margin={"l": 85, "r": 35, "t": 80, "b": 95},
        legend={
            "orientation": "h",
            "yanchor": "bottom",
            "y": legend_y,
            "xanchor": "center",
            "x": 0.5,
            "font": {"family": "Times New Roman", "size": 14},
        },
    )
    fig.update_xaxes(
        title={"text": x_title, "font": {"family": "Times New Roman", "size": 14}},
        showgrid=True,
        gridcolor="#D9D9D9",
        griddash="dot",
        zeroline=False,
        linecolor="black",
        mirror=True,
        ticks="outside",
        tickfont={"family": "Times New Roman", "size": 14},
    )
    fig.update_yaxes(
        title={"text": y_title or "", "font": {"family": "Times New Roman", "size": 14}},
        showgrid=True,
        gridcolor="#D9D9D9",
        griddash="dot",
        zeroline=False,
        linecolor="black",
        mirror=True,
        ticks="outside",
        tickfont={"family": "Times New Roman", "size": 14},
    )
    return fig


def save_plotly(fig, stem):
    html = OUT_DIR / f"{stem}.html"
    png = OUT_DIR / f"{stem}.png"
    fig.write_html(html, include_plotlyjs="cdn")
    fig.write_image(png, scale=2)
    return png, html


def metric_values_text(df, metrics, zones=ZONE_ORDER):
    parts = []
    for z in zones:
        row = df.loc[z]
        vals = ", ".join(f"{metric}={row[metric]:.3f}" for metric in metrics)
        parts.append(f"{ZONE_LABELS[z]}: {vals}")
    return "; ".join(parts) + "."


In [ ]:
# Функция строит основные графики для статьи и сохраняет их в файлы.
def build_figures():
    figures = []
    x_labels = [ZONE_LABELS[z] for z in ZONE_ORDER]

    fig1 = go.Figure()
    for model in MODEL_ORDER:
        m = zone_model[zone_model["model"] == model].set_index("latitude_zone")
        fig1.add_trace(go.Scatter(
            x=x_labels,
            y=[m.loc[z, "rmse_mean"] for z in ZONE_ORDER],
            mode="lines+markers+text",
            name=model,
            text=[f"{m.loc[z, 'rmse_mean']:.2f}" for z in ZONE_ORDER],
            textposition="top center",
            line={"width": 2, "color": COLORS[model]},
            marker={"size": 9, "symbol": "circle", "line": {"width": 1, "color": "black"}},
        ))
    gost_layout(fig1, "??????? RMSE ??????? ?? ???????? ?????", "RMSE foF2, ???")
    figures.append(("rmse_all", *save_plotly(fig1, "gost_plotly_rmse_by_zone_model")))

    fig2 = go.Figure()
    for model in NONLINEAR_MODELS:
        m = zone_model[zone_model["model"] == model].set_index("latitude_zone")
        fig2.add_trace(go.Scatter(
            x=x_labels,
            y=[m.loc[z, "rmse_mean"] for z in ZONE_ORDER],
            mode="lines+markers+text",
            name=model,
            text=[f"{m.loc[z, 'rmse_mean']:.2f}" for z in ZONE_ORDER],
            textposition="top center",
            line={"width": 2, "color": COLORS[model]},
            marker={"size": 10, "symbol": "circle", "line": {"width": 1, "color": "black"}},
        ))
    gost_layout(fig2, "??????? RMSE ?????????? ??????? ?? ???????? ?????", "RMSE foF2, ???")
    figures.append(("rmse_nonlinear", *save_plotly(fig2, "gost_plotly_rmse_nonlinear_by_zone")))

    fig3 = make_subplots(
        rows=2,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.13,
        subplot_titles=("?????? ????????", "??????????????? ???????? ? ??????????"),
    )
    zdf = zone_summary_nl.loc[ZONE_ORDER]
    for metric, marker in [("mae_mean", "circle"), ("rmse_mean", "square")]:
        label = "MAE" if metric == "mae_mean" else "RMSE"
        fig3.add_trace(go.Scatter(
            x=x_labels,
            y=zdf[metric],
            mode="lines+markers+text",
            name=label,
            text=[f"{v:.2f}" for v in zdf[metric]],
            textposition="top center",
            line={"width": 2, "color": COLORS[label]},
            marker={"size": 9, "symbol": marker, "line": {"width": 1, "color": "black"}},
        ), row=1, col=1)
    for metric, label, marker in [("r2_mean", "R?", "diamond"), ("corr_mean", "??????????", "triangle-up")]:
        fig3.add_trace(go.Scatter(
            x=x_labels,
            y=zdf[metric],
            mode="lines+markers+text",
            name=label,
            text=[f"{v:.2f}" for v in zdf[metric]],
            textposition="top center",
            line={"width": 2, "color": COLORS[label]},
            marker={"size": 9, "symbol": marker, "line": {"width": 1, "color": "black"}},
        ), row=2, col=1)
    gost_layout(fig3, "??????? ???????? ???????? ?? ???????? ?????", legend_y=-0.22)
    fig3.update_yaxes(title_text="MAE, RMSE, ???", row=1, col=1)
    fig3.update_yaxes(title_text="R?, ??????????", row=2, col=1)
    fig3.update_layout(height=780)
    figures.append(("quality_profile", *save_plotly(fig3, "gost_plotly_quality_profile_by_zone")))

    fig4 = go.Figure()
    best = station_model.loc[station_model.groupby("station")["rmse"].idxmin()].copy()
    counts = best.groupby(["latitude_zone", "model"]).size().unstack(fill_value=0).reindex(ZONE_ORDER).fillna(0)
    for model in ["RandomForest", "XGBoost", "CatBoost"]:
        y = [counts.loc[z, model] if model in counts.columns else 0 for z in ZONE_ORDER]
        fig4.add_trace(go.Scatter(
            x=x_labels,
            y=y,
            mode="lines+markers+text",
            name=model,
            text=[str(int(v)) for v in y],
            textposition="top center",
            line={"width": 2, "color": COLORS[model]},
            marker={"size": 10, "symbol": "circle", "line": {"width": 1, "color": "black"}},
        ))
    gost_layout(fig4, "?????????? ??????? ? ??????????? RMSE ?? ???????", "????? ???????")
    figures.append(("best_model_counts", *save_plotly(fig4, "gost_plotly_best_model_counts_by_zone")))
    return figures

figures = build_figures()
figures


In [ ]:
# Выполнение расчетного шага и вывод промежуточного результата.
# ????????? ??????? ??? ?????? ??? ?????????.
zdf = zone_summary_nl.loc[ZONE_ORDER]
figure_notes = {
    "rmse_all": "???????? RMSE ??? ???? ??????? ????????? ?? ???????; ??????? ???????? ????? LinearRegression.",
    "rmse_nonlinear": metric_values_text(zdf.rename(columns={"rmse_mean": "RMSE"}), ["RMSE"]),
    "quality_profile": metric_values_text(zdf.rename(columns={"mae_mean": "MAE", "rmse_mean": "RMSE", "r2_mean": "R?", "corr_mean": "corr"}), ["MAE", "RMSE", "R?", "corr"]),
}
figure_notes


In [ ]:
# Функции ниже отвечают за оформление Word-документа и добавление материалов статьи.
def set_document_style(doc):
    style = doc.styles["Normal"]
    style.font.name = "Times New Roman"
    style.font.size = Pt(14)
    for section in doc.sections:
        section.top_margin = Cm(2)
        section.bottom_margin = Cm(2)
        section.left_margin = Cm(3)
        section.right_margin = Cm(1.5)


def add_paragraph(doc, text, align=WD_ALIGN_PARAGRAPH.JUSTIFY, first_line=True, size=14):
    p = doc.add_paragraph()
    p.alignment = align
    if first_line:
        p.paragraph_format.first_line_indent = Cm(1.25)
    p.paragraph_format.line_spacing = 1.0
    run = p.add_run(text)
    run.font.name = "Times New Roman"
    run.font.size = Pt(size)
    return p


def add_center(doc, text, bold=False, size=14):
    p = doc.add_paragraph()
    p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    run = p.add_run(text)
    run.bold = bold
    run.font.name = "Times New Roman"
    run.font.size = Pt(size)
    return p


def add_table(doc, data, columns, headers):
    table = doc.add_table(rows=1, cols=len(columns))
    table.style = "Table Grid"
    for i, header in enumerate(headers):
        table.rows[0].cells[i].text = header
    for _, row in data.iterrows():
        cells = table.add_row().cells
        for i, col in enumerate(columns):
            cells[i].text = str(row[col])
    for row in table.rows:
        for cell in row.cells:
            for paragraph in cell.paragraphs:
                for run in paragraph.runs:
                    run.font.name = "Times New Roman"
                    run.font.size = Pt(12)
    return table


In [ ]:
# Функция собирает итоговый Word-документ из текста, таблиц и графиков.
def build_article_docx():
    doc = Document()
    set_document_style(doc)

    add_paragraph(doc, "??? 004.85:550.388", WD_ALIGN_PARAGRAPH.LEFT, first_line=False)
    add_center(doc, "????????? ??????? ?????????????")
    add_center(doc, "??????? ????????????: ?????? ?????? ?????????????, ??. ????. ??????? ???????????? ? ?????")
    add_center(doc, "????? ?? ??????????? ??????????????? ??????????????? ????????????, ?. ??????-???")
    add_center(doc, "??????? ???????? ???? ??????????? ??????? ?? ???????? 24-???????? ML-???????? ??????????? ??????? ???? F2", bold=True)

    add_paragraph(doc, "?????????. ??????????? ??????? ????????? ????????? ??????????? ??????? ?? ???????? 24-???????? ???????? ??????????? ??????? ???? F2 (foF2). ??? 35 ??????? ???? GIRO ????????? ????????? ??????? Linear Regression, ElasticNet, Random Forest, XGBoost ? CatBoost. ???????? ???????? ??????????? ?? MAE, RMSE, R? ? ???????????? ??????????. ??????? ???? ???????????? ?? ??????? ???????? ?????: ?????? ??????, ????? ??????? ??????, ???????? ??????? ?????? ? ???????? ??????? ??????. ?? ??????? ????????? ?????? ??? ?????????? ??????? ???????? ??????????? ???????? RMSE ? ????? ? ???????? ??????? ???????: 1,000 ? 1,036 ??? ??????????????.")
    add_paragraph(doc, "???????? ?????: foF2, ?????????, ??????????? ???????, ???????? ????????, ???????? ????, Random Forest, XGBoost, CatBoost.", WD_ALIGN_PARAGRAPH.LEFT, first_line=False)

    paragraphs = [
        "??????????? ??????? ???? F2 (foF2) ????????? ? ???????? ??????????, ??????????? ????????? ????????? ? ??????? ??????????????? ???????? ?????????. ???????? foF2 ?????????? ???????????? ??????? ????????????, ????????????? ?? ???? F2 ??? ???????????? ???????. ??????? ??????????????? foF2 ??????????? ??? ?????? ??????? ??????????, ????????? ? ??????????? ??????????? ??????.",
        "???????? ????????? ??????? ?????? ???????? ? ??????? ??????????? ????????????. ? ?????? ??????? ?? foF2 ?????? ?????????????? ??????????? ????????. ? ??????? ??????? ??????? ????? ??????????? ????????? ? ???????????? ??????????. ? ??????? ??????? ???????? ? ???????? ?????????? ???????? ?????????? ???????????? ????????????? ? ?????????????? ????????. ??????? ???????? 24-???????? ???????? ????????????? ????????????? ???????? ?? ???????? ?????.",
        "???? ?????? - ??????? ??????? ???????? ???? ??????????? ??????? ?? ???????? 24-???????? ML-???????? foF2.",
        "??? ?????????? ???? ????????? ??????? ?????????? ??????? ????????? ????????, ???????? ??????? ???????? ??? ?????? ???????, ????????? ????????????? ??????? ?? ???????? ?????, ???????????? ??????? ?????? ? ????????? ????????? MAE, RMSE, R? ? ?????????? ????? ????????? ??????.",
        "? ???????????? ???????????? ?????? ??????????? ??????? ?? 2024-2025 ???? ? ????????? ????? 15 ?????. ??????? ???????? ?? ???????? 24 ????. ? ???????? ??????? ???????????? Linear Regression, ElasticNet, Random Forest, XGBoost ? CatBoost. ??????????? ?????????? ? ?????? walk-forward daily. ??? ?????? ??????????? MAE, RMSE, R?, ??????????? ?????????? ? MAPE.",
    ]
    for text in paragraphs:
        add_paragraph(doc, text)

    station_zone = metadata.groupby("latitude_zone").agg(
        stations=("station", "nunique"), lat_min=("latitude", "min"), lat_max=("latitude", "max")
    ).reindex(ZONE_ORDER).reset_index()
    station_zone["latitude_zone"] = station_zone["latitude_zone"].map(ZONE_LABELS)
    station_zone["lat_min"] = station_zone["lat_min"].map(lambda x: f"{x:.2f}")
    station_zone["lat_max"] = station_zone["lat_max"].map(lambda x: f"{x:.2f}")
    add_center(doc, "??????? 1 - ????????????? ??????? ?? ???????? ?????")
    add_table(doc, station_zone, ["latitude_zone", "stations", "lat_min", "lat_max"], ["???????? ????", "????? ???????", "???. ??????", "????. ??????"])

    table2 = zone_summary_nl.loc[ZONE_ORDER].reset_index()
    table2["latitude_zone"] = table2["latitude_zone"].map(ZONE_LABELS)
    for col in ["mae_mean", "rmse_mean", "r2_mean", "corr_mean"]:
        table2[col] = table2[col].map(lambda x: f"{x:.3f}")
    add_center(doc, "??????? 2 - ??????? ??????? ?? ???????? ????? ??? ?????????? ???????")
    add_table(doc, table2, ["latitude_zone", "stations", "mae_mean", "rmse_mean", "r2_mean", "corr_mean"], ["???????? ????", "????? ???????", "MAE", "RMSE", "R?", "??????????"])

    add_paragraph(doc, "?? ??????? 2 ??? ?????????? ??????? ? ???? S_mid ???????? MAE=0,809 ???, RMSE=1,000 ???, R?=0,393 ? ??????????=0,900. ? ???? N_mid ???????? MAE=0,808 ???, RMSE=1,036 ???, R?=0,437 ? ??????????=0,882. ? ???? Low ???????? ????????? MAE=1,170 ???, RMSE=1,508 ???, R?=-0,429 ? ??????????=0,866. ? ???? N_high ???????? ????????? MAE=0,865 ???, RMSE=1,052 ???, R?=-0,399 ? ??????????=0,510; ???? ???????? 2 ???????.")

    fig_map = {name: png for name, png, html in figures}
    captions = [
        ("rmse_all", "???. 1. ??????? RMSE ??????? ????????? ???????? ?? ???????? ?????.", "?? ??????? ???????? ??????? LinearRegression ?? ??????? RMSE: Low=9,29 ???, S_mid=4,44 ???, N_mid=5,94 ???, N_high=7,85 ??? ??? LinearRegression."),
        ("rmse_nonlinear", "???. 2. ??????? RMSE ?????????? ??????? ?? ???????? ?????.", figure_notes["rmse_nonlinear"]),
        ("quality_profile", "???. 3. ??????? ???????? ???????? ?? ???????? ????? ??? ?????????? ???????.", figure_notes["quality_profile"]),
        ("best_model_counts", "???. 4. ?????????? ??????? ? ??????????? RMSE ?? ??????? ? ???????? ?????.", "CatBoost: Low=8, S_mid=3, N_mid=10, N_high=2; RandomForest: Low=3, S_mid=1, N_mid=8, N_high=0; XGBoost: Low=0, S_mid=0, N_mid=0, N_high=0.")
    ]
    for key, caption, note in captions:
        doc.add_picture(str(fig_map[key]), width=Cm(15.5))
        add_center(doc, caption)
        add_paragraph(doc, "????????: " + note, first_line=False)

    add_paragraph(doc, "??? ????????? ??????? ?? ????????? ???????? ??????????? RMSE ???? ???????????? ? CatBoost: 8 ??????? ? ???? Low, 3 ??????? ? ???? S_mid, 10 ??????? ? ???? N_mid ? 2 ??????? ? ???? N_high. ??? Random Forest ??????????? RMSE ??????? ?? 3 ???????? ???? Low, 1 ??????? ???? S_mid ? 8 ???????? ???? N_mid. ??? XGBoost ??????????? RMSE ????? ??????? ? ?????? ??????? ?? ????????????.")
    add_paragraph(doc, "?????????? ????????????? ???????? ??????? ? ???????? ?????????? ?????????. ? ??????? ??????? ???????? ? ???????? ???????????? foF2 ????????? ??????????? ??????, ??? ?????????? ? RMSE ????? 1,0 ??? ? ?????????? 0,88-0,90. ? ?????? ??????? ?????????? MAE ? RMSE ??????? ? ?????????????? ??????????? ????????? ? ?????????? ???????????????? ??????????? ????????????. ? ???????? ??????? ??????? ???????? ?????????? 0,510 ??????? ? ??????? ??????????? ????????? ? ???????????? ??????????; ??-?? ???? ??????? ? ???? ????? ????? ??????????? ?? ?????? ???????.")
    add_paragraph(doc, "??????????. ????????? ????????? ???????? 24-???????? ML-???????? foF2 ??? 35 ??????????? ???????, ?????????????? ?? ???????? ?????. ??? ?????????? ??????? ??????????? ??????? ???????? RMSE ???????? ? ????? S_mid ? N_mid: 1,000 ? 1,036 ???. ? ???? Low ??????? RMSE ???????? 1,508 ???. ? ???? N_high ??????? RMSE ???????? 1,052 ??? ??? ?????????? 0,510 ? ????? ???????, ?????? 2. ?? ????? ??????? ? ??????????? RMSE ?????????? ?????????? ??????? ????????? ? ?????? CatBoost.")

    add_paragraph(doc, "?????? ??????????:", WD_ALIGN_PARAGRAPH.LEFT, first_line=False)
    refs = [
        "1. Bilitza D. International Reference Ionosphere 2020 / D. Bilitza, D. Altadill, V. Truhlik, D. Buresova, I. Galkin // Reviews of Geophysics. - 2022. - Vol. 60, Iss. 4. - DOI: 10.1029/2021RG000753.",
        "2. Razin B. V. Global Ionospheric Radio Observatory (GIRO) / B. V. Razin, I. A. Galkin // Earth, Planets and Space. - 2011. - Vol. 63. - P. 377-381. - DOI: 10.5047/eps.2011.03.001.",
        "3. Mao S. A Review of Machine Learning-Based Ionospheric Spatial and Temporal Modeling / S. Mao, M. Hern?ndez-Pajares, B. Soja // Journal of Geophysical Research: Space Physics. - 2024. - DOI: 10.1029/2024JH000555.",
        "4. ???????? ?. ?. ????????????? ??????? ??????????? ??????? ???????????? ???? F2 / ?. ?. ????????, ?. ?. ?????????, ?. ?. ????????, ?. ?. ????????, ?. ?. ?????? // ?????????? ????????? ????? ? ??????????????????? ??????????, ??????? ? ??????? ?????????????????????? ?????????. - 2022. - ? 49. - ?. 131-146.",
    ]
    for ref in refs:
        add_paragraph(doc, ref)

    article_path = OUT_DIR / "??????_foF2_????????_????_????.docx"
    doc.save(article_path)
    return article_path

article_path = build_article_docx()
article_path


In [ ]:
# Выполнение расчетного шага и вывод промежуточного результата.
# ??????? ???????? ??????????.
doc = Document(str(article_path))
print("DOCX:", article_path)
print("??????:", len(doc.paragraphs))
print("???????:", len(doc.tables))
print("???????:", len(doc.inline_shapes))
for name, png, html in figures:
    print(name, png.name, html.name)
